In [1]:
!pip install transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 86.3 MB/s eta 0:00:00:00:01:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.

In [4]:
import os
from datasets import load_dataset
from transformers import (
    PreTrainedTokenizerFast,
    BertConfig,
    BertForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

def run_bert_pilot_training():

    
    dataset_path = "/kaggle/input/datasets/romaincom/ancient-greek-corpus/metadata_ancient_greek_corpus.jsonl"
    tokenizer_path = "/kaggle/input/datasets/romaincom/ancient-greek-corpus/ancient_greek_bpe.json"
    
    # Le dossier de sortie où le modèle sera sauvegardé
    output_dir = "/kaggle/working/ancient-greek-bert"

    print("Chargement du Tokenizer BPE...")
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_file=tokenizer_path,
        unk_token="[UNK]",
        pad_token="[PAD]",
        cls_token="[CLS]",
        sep_token="[SEP]",
        mask_token="[MASK]"
    )

    # ---------------------------------------------------------
    # 2. PRÉPARATION DES DONNÉES (Dataset & Split)
    # ---------------------------------------------------------
    print("Chargement du corpus en mémoire (via la librairie HuggingFace Datasets)...")
    raw_datasets = load_dataset("json", data_files=dataset_path)
    
    # Séparation : 95% pour l'entraînement, 5% pour l'évaluation (Dev Loss)
    split_datasets = raw_datasets["train"].train_test_split(test_size=0.05, seed=42)
    
    def tokenize_function(examples):
        # On découpe les phrases. max_length=128 est parfait pour un "Pilot Run"
        return tokenizer(examples["text"], return_special_tokens_mask=True, truncation=True, max_length=128)

    print("Tokenisation du corpus en cours (sur tous les cœurs CPU)...")
    tokenized_datasets = split_datasets.map(
        tokenize_function,
        batched=True,
        num_proc=4, # Utilise 4 cœurs du CPU Kaggle
        remove_columns=raw_datasets["train"].column_names # On ne garde que les vecteurs mathématiques
    )

    # ---------------------------------------------------------
    # 3. ARCHITECTURE DU MODÈLE BERT
    # ---------------------------------------------------------
    print("Initialisation du modèle BERT vierge...")
    config = BertConfig(
        vocab_size=tokenizer.vocab_size,
        hidden_size=768,       # Architecture standard de BERT Base
        num_hidden_layers=12,
        num_attention_heads=12,
        max_position_embeddings=512,
    )
    model = BertForMaskedLM(config=config)
    
    print(f"Taille du modèle : {model.num_parameters() / 1_000_000:.1f} millions de paramètres.")

    # ---------------------------------------------------------
    # 4. CONFIGURATION DE L'ENTRAÎNEMENT (Optimisé Dual T4)
    # ---------------------------------------------------------
    # Le Data Collator va masquer 15% des mots (mlm_probability=0.15)
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=True, mlm_probability=0.15
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="steps",
        eval_steps=1000,            # Passe un examen toutes les 1000 étapes
        save_steps=1000,            # Sauvegarde le modèle toutes les 1000 étapes
        logging_steps=100,          # Affiche les logs toutes les 100 étapes
        learning_rate=1e-4,         # Taux d'apprentissage standard pour du pre-training
        per_device_train_batch_size=32, # Batch massif grâce aux deux T4
        per_device_eval_batch_size=32,
        fp16=True,                  # Accélération matérielle (Tensor Cores)
        max_steps=5000,             # PILOT RUN : On arrête au bout de 5000 étapes pour vérifier
        report_to="none"            # Désactive les outils externes (Weights & Biases)
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
    )

    print("Début de l'entraînement Pilot Run MLM ")
    trainer.train()
    
    # Sauvegarde finale
    trainer.save_model(os.path.join(output_dir, "final_pilot_model"))
    print(" Entraînement terminé et modèle sauvegardé !")

if __name__ == "__main__":
    run_bert_pilot_training()

Chargement du Tokenizer BPE...
Chargement du corpus en mémoire (via la librairie HuggingFace Datasets)...
Tokenisation du corpus en cours (sur tous les cœurs CPU)...
Initialisation du modèle BERT vierge...
Taille du modèle : 109.1 millions de paramètres.
🚀 Début de l'entraînement Pilot Run MLM 🚀


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
1000,16.122482,16.117121
2000,16.069169,16.024788
3000,16.029785,15.987500
4000,15.979402,15.968216
5000,15.946022,15.963215


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Entraînement terminé et modèle sauvegardé !
